## **<font color="red">Linear Regression</font>**

In [1]:
from sklearn.datasets import load_diabetes
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# dataset
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model
reg = LinearRegression()
reg.fit(X_train, y_train)
print("coef_: ", reg.coef_)
print("intercept_: ", reg.intercept_)

# prediction
y_pred = reg.predict(X_test)
r2_score(y_test, y_pred)

coef_:  [  37.90402135 -241.96436231  542.42875852  347.70384391 -931.48884588
  518.06227698  163.41998299  275.31790158  736.1988589    48.67065743]
intercept_:  151.34560453985995


0.4526027629719197

## **<font color="red">Batch Gradient Descent</font>**

In [2]:
from sklearn.datasets import load_diabetes
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# model class
class GDRegressor:
    def __init__(self, lr=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = lr
        self.epochs = epochs

    def fit(self, X_train, y_train):
        # init your coefs
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])

        for i in range(self.epochs):
            # update all the coef and the intercept
            # Vectorization
            y_hat = np.dot(X_train, self.coef_) + self.intercept_
            intercept_der = -2 * np.mean(y_train - y_hat)
            self.intercept_ = self.intercept_ - (self.lr * intercept_der)

            coef_der = -2 * np.dot((y_train - y_hat), X_train)
            self.coef_ = self.coef_ - (self.lr * coef_der)
        
        print(self.intercept_, self.coef_)

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_


# dataset
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model
gdr = GDRegressor(epochs=150, lr=0.01)
gdr.fit(X_train, y_train)
print("coef_: ", gdr.coef_)
print("intercept_: ", gdr.intercept_)

# prediction
y_pred = gdr.predict(X_test)
r2_score(y_test, y_pred)



144.08720181553275 [  54.72131955 -212.57644092  527.11032489  324.91251501  -82.13769161
 -133.54889058 -220.1532492   139.55513261  383.65846275  122.23536977]
coef_:  [  54.72131955 -212.57644092  527.11032489  324.91251501  -82.13769161
 -133.54889058 -220.1532492   139.55513261  383.65846275  122.23536977]
intercept_:  144.08720181553275


0.4325525922094896

### **<font color="blue">Batch Gradient Descent Animation</font>**

In [3]:
%matplotlib notebook

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from mpl_toolkits.mplot3d import Axes3D  # registers 3D projection
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# --------------------------
# Simple GDRegressor (records history)
# --------------------------
class GDRegressor:
    def __init__(self, lr=0.01, epochs=100, verbose=False):
        self.lr = lr
        self.epochs = int(epochs)
        self.verbose = verbose
        self.coef_ = None
        self.intercept_ = None
        self.history = {'coef': [], 'intercept': [], 'loss': []}

    def _loss(self, X, y, coef, intercept):
        return float(np.mean((y - (X.dot(coef) + intercept)) ** 2))

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).ravel()
        n, d = X.shape
        # init
        self.coef_ = np.zeros(d, dtype=float)
        self.intercept_ = 0.0
        for epoch in range(self.epochs):
            preds = X.dot(self.coef_) + self.intercept_
            error = y - preds

            # gradients (MSE)
            d_intercept = -2.0 * np.mean(error)
            d_coef = -2.0 * (X.T.dot(error)) / n

            self.intercept_ -= self.lr * d_intercept
            self.coef_ -= self.lr * d_coef

            loss = self._loss(X, y, self.coef_, self.intercept_)
            self.history['coef'].append(self.coef_.copy())
            self.history['intercept'].append(float(self.intercept_))
            self.history['loss'].append(loss)

            if self.verbose and (epoch % max(1, self.epochs // 10) == 0):
                print(f"Epoch {epoch:4d}/{self.epochs} loss={loss:.4f}")

    def predict(self, X):
        return X.dot(self.coef_) + self.intercept_

# --------------------------
# Prepare diabetes dataset (use first two features for visualization)
# --------------------------
X_all, y_all = load_diabetes(return_X_y=True)
X = X_all[:, :2]        # first two features only (so we can plot plane)
y = y_all.copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------
# Train model
# --------------------------
model = GDRegressor(lr=0.05, epochs=200, verbose=True)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print("\nR2 score on test set:", r2_score(y_test, y_pred))

# --------------------------
# Prepare animation data
# --------------------------
# We'll plot plane over original-feature-scale grid (for interpretability),
# but predictions use scaled features internally.
X_train_orig = scaler.inverse_transform(X_train_scaled)
xs = X_train_orig[:, 0]
ys = X_train_orig[:, 1]
zs = y_train

# grid (original scale) for plotting plane surface
xx = np.linspace(xs.min(), xs.max(), 30)
yy = np.linspace(ys.min(), ys.max(), 30)
XX, YY = np.meshgrid(xx, yy)
grid_orig = np.column_stack([XX.ravel(), YY.ravel()])
grid_scaled = scaler.transform(grid_orig)  # convert to scaled before predicting

# history arrays (coefs, intercepts, losses)
coef_hist = np.array(model.history['coef'])        # shape (epochs, n_features=2)
intercept_hist = np.array(model.history['intercept'])
loss_hist = np.array(model.history['loss'])

# choose frame step (animate every `step` epochs) and ensure last epoch included
step = 10
frame_indices = np.arange(0, len(coef_hist), step)
if frame_indices[-1] != len(coef_hist) - 1:
    frame_indices = np.append(frame_indices, len(coef_hist) - 1)

# --------------------------
# Setup figure & artists
# --------------------------
fig = plt.figure(figsize=(10, 4))
ax3d = fig.add_subplot(1, 2, 1, projection='3d')
ax_contour = fig.add_subplot(1, 2, 2)

# scatter training points
ax3d.scatter(xs, ys, zs, s=18, alpha=0.6)
ax3d.set_xlabel('Feature 0 (orig)')
ax3d.set_ylabel('Feature 1 (orig)')
ax3d.set_zlabel('Target (y)')
ax3d.set_title('3D: data + evolving fitted plane')

# initial (placeholder) surface artist reference
surface_ref = [None]

# contour: compute a static contour of loss over w0,w1 using final intercept (for visualization)
w0_vals = np.linspace(coef_hist[:,0].min()-1, coef_hist[:,0].max()+1, 120)
w1_vals = np.linspace(coef_hist[:,1].min()-1, coef_hist[:,1].max()+1, 120)
W0, W1 = np.meshgrid(w0_vals, w1_vals)
loss_grid = np.zeros_like(W0)
intercept_fixed = intercept_hist[-1]
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        w = np.array([W0[i,j], W1[i,j]])
        preds_grid = X_train_scaled.dot(w) + intercept_fixed
        loss_grid[i,j] = np.mean((y_train - preds_grid) ** 2)

cs = ax_contour.contour(W0, W1, loss_grid, levels=30)
ax_contour.set_xlabel('w0 (scaled)')
ax_contour.set_ylabel('w1 (scaled)')
ax_contour.set_title('Loss contour (intercept fixed) and GD path')

# path artists
path_line, = ax_contour.plot([], [], 'r-', lw=2)
current_point, = ax_contour.plot([], [], 'ko', markersize=6)

# 3D path artists (line + current point)
line3d, = ax3d.plot([], [], [], color='red', lw=2, label='GD path')
point3d, = ax3d.plot([], [], [], 'ko', markersize=5)

ax_contour.legend()
ax3d.legend()

# --------------------------
# Animation update function
# --------------------------
def update_frame(idx_in_frames):
    """
    idx_in_frames: index into frame_indices array (not epoch number directly).
    We compute epoch = frame_indices[idx_in_frames] and update artists accordingly.
    """
    epoch = int(frame_indices[idx_in_frames])
    coef = coef_hist[epoch]            # length-2 array
    intercept = float(intercept_hist[epoch])
    loss = float(loss_hist[epoch])

    # compute plane z values on the grid (use scaled grid for predictions)
    z_grid = grid_scaled.dot(coef) + intercept
    Z = z_grid.reshape(XX.shape)

    # remove previous surface if exists
    if surface_ref[0] is not None:
        try:
            surface_ref[0].remove()
        except Exception:
            pass

    # plot new surface and keep reference
    surface_ref[0] = ax3d.plot_surface(XX, YY, Z, alpha=0.45, linewidth=0, antialiased=True)

    # update 3D path (w0 over epochs, w1 over epochs, loss over epochs)
    # we will plot the path of the two weights vs loss (z axis)
    w0_path = coef_hist[:epoch+1, 0]
    w1_path = coef_hist[:epoch+1, 1]
    z_path = loss_hist[:epoch+1]

    line3d.set_data(w0_path, w1_path)
    line3d.set_3d_properties(z_path)
    # point must receive sequences for set_data
    point3d.set_data([w0_path[-1]], [w1_path[-1]])
    point3d.set_3d_properties([z_path[-1]])

    # update contour path and current point (sequences)
    path_line.set_data(w0_path, w1_path)
    current_point.set_data([w0_path[-1]], [w1_path[-1]])

    # update titles to show epoch and loss
    ax3d.set_title(f'3D: epoch {epoch+1}/{len(coef_hist)}  loss={loss:.4f}')
    ax_contour.set_title(f'Contour: epoch {epoch+1}/{len(coef_hist)}  loss={loss:.4f}')

    return [surface_ref[0], line3d, point3d, path_line, current_point]

# Create animation using frame_indices as frames
n_frames = len(frame_indices)
anim = animation.FuncAnimation(fig, update_frame, frames=range(n_frames), interval=100, blit=False)

# Try robust display: prefer to_jshtml (works in many JupyterLab setups), fallback to html5 video, then GIF, then plt.show()
try:
    html = anim.to_jshtml()   # first try: JS animation (works in many notebook/lab setups)
    display(HTML(html))
except Exception as e_js:
    print("to_jshtml() failed:", e_js)
    try:
        print("Trying to embed HTML5 video (requires ffmpeg).")
        vid_html = anim.to_html5_video()
        display(HTML(vid_html))
    except Exception as e_vid:
        print("to_html5_video() failed:", e_vid)
        # Attempt to save GIF via PillowWriter (pillow required)
        try:
            from matplotlib.animation import PillowWriter
            gif_path = "gd_training_fallback.gif"
            print("Saving as GIF (this may take a few seconds)...")
            anim.save(gif_path, writer=PillowWriter(fps=15))
            display(HTML(f'<img src="{gif_path}" />'))
        except Exception as e_gif:
            print("Saving GIF failed:", e_gif)
            print("Final fallback: display static figure and call plt.show().")
            plt.show()



Epoch    0/200 loss=25176.0750
Epoch   20/200 loss=6124.3668
Epoch   40/200 loss=5841.6449
Epoch   60/200 loss=5837.4147
Epoch   80/200 loss=5837.3504
Epoch  100/200 loss=5837.3494
Epoch  120/200 loss=5837.3494
Epoch  140/200 loss=5837.3494
Epoch  160/200 loss=5837.3494
Epoch  180/200 loss=5837.3494

R2 score on test set: -0.0023337254494764093


<IPython.core.display.Javascript object>

C:\Users\Vikas\AppData\Local\Temp\ipykernel_6164\1471555975.py:147: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_contour.legend()
